<a href="https://colab.research.google.com/github/Kevinlo937/cord-cutting-predicated/blob/main/topMSO_ML_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ✅ 先安裝 imbalanced-learn（若在 Colab 尚未安裝）
!pip install imbalanced-learn

# ✅ 匯入必要套件
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE



In [ ]:
# ✅ 讀取資料（請將路徑換成自己的）
myDatasetPath='csr_service_bill.csv'
df = pd.read_csv(myDatasetPath)

# ✅ 資料清理與預處理
df = df.dropna().reset_index(drop=True)
y = df["使用狀態_數值"]

# 指定排除欄位
exclude_features = ['CUST_NO', '相關編號', '使用狀態_數值']

# 選擇 X 的特徵欄位 (排除指定欄位)
X = df.drop(columns=exclude_features)

# 數值標準化
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# ✅ 使用 SMOTE 平衡樣本
X_smote, y_smote = SMOTE(random_state=42).fit_resample(X_scaled, y)

# ✅ 資料切分
X_train, X_test, y_train, y_test = train_test_split(X_smote, y_smote, test_size=0.2, random_state=42, stratify=y_smote)

# ✅ 對 SVM 使用 GridSearchCV 找最佳參數
'''
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 0.01, 0.1, 1],
    'kernel': ['rbf'],
    'class_weight': ['balanced']
}
# found Best SVM Params: {'C': 10, 'class_weight': 'balanced', 'gamma': 'scale', 'kernel': 'rbf'}

param_grid = {
    'C': [1, 10, 50, 100],
    'gamma': [0.1, 0.5, 1, 5],
    'kernel': ['rbf'],
    'class_weight': ['balanced']
}
# found Best SVM Params: {'C': 100, 'class_weight': 'balanced', 'gamma': 5, 'kernel': 'rbf'}
'''
'''
param_grid = {
    'C': [0.1, 1, 10, 100, 1000],
    'gamma': [0.0001, 0.001, 0.01, 0.1, 1],
    'kernel': ['rbf'],
}
# spent 21 hours more and not yet finish...
'''
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 1],
    'kernel': ['rbf'],
}
# Best SVM Params: {'C': 10, 'gamma': 1, 'kernel': 'rbf'}
# cv=10, instead of 3
grid_search = GridSearchCV(SVC(), param_grid, cv=3, scoring='f1_macro', n_jobs=-1, verbose=30)  # 設定 verbose=3
grid_search.fit(X_train, y_train)

best_svm = grid_search.best_estimator_
print("Best SVM Params:", grid_search.best_params_)

# ✅ 定義模型字典
models = {
    "KNN (k=5, SMOTE)": KNeighborsClassifier(n_neighbors=5),
    "SVM (GridSearch, SMOTE)": best_svm,
    "Random Forest (SMOTE)": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
}

# ✅ 訓練與預測
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {
        "confusion": confusion_matrix(y_test, y_pred),
        "report": classification_report(y_test, y_pred, output_dict=True)
    }

# ✅ 成果摘要表
summary = []
for name, result in results.items():
    report = result["report"]
    # Get the actual labels from the report
    labels = list(report.keys())
    # Assuming '0' and '1' are the first two labels, but handle other cases
    label_0 = labels[0] if labels[0] != 'accuracy' else labels[1]
    label_1 = labels[1] if labels[1] != 'accuracy' else labels[2]

    row = {
        "Model": name,
        "Accuracy": report["accuracy"],
        "Precision_0": report[label_0]["precision"],
        "Recall_0": report[label_0]["recall"],
        "F1_0": report[label_0]["f1-score"],
        "Precision_1": report[label_1]["precision"],
        "Recall_1": report[label_1]["recall"],
        "F1_1": report[label_1]["f1-score"]
    }
    summary.append(row)

df_summary = pd.DataFrame(summary)
print("\n📊 模型效能比較：")
print(df_summary)


Fitting 3 folds for each of 16 candidates, totalling 48 fits
Best SVM Params: {'C': 100, 'gamma': 1, 'kernel': 'rbf'}


KeyError: '0'

In [ ]:
# ✅ 成果摘要表
summary = []
for name, result in results.items():
    report = result["report"]
    # Get the actual labels from the report
    labels = list(report.keys())
    # Assuming '0' and '1' are the first two labels, but handle other cases
    label_0 = labels[0] if labels[0] != 'accuracy' else labels[1]
    label_1 = labels[1] if labels[1] != 'accuracy' else labels[2]

    row = {
        "Model": name,
        "Accuracy": report["accuracy"],
        "Precision_0": report[label_0]["precision"],
        "Recall_0": report[label_0]["recall"],
        "F1_0": report[label_0]["f1-score"],
        "Precision_1": report[label_1]["precision"],
        "Recall_1": report[label_1]["recall"],
        "F1_1": report[label_1]["f1-score"]
    }
    summary.append(row)

df_summary = pd.DataFrame(summary)
print("\n📊 模型效能比較：")
print(df_summary)


📊 模型效能比較：
                     Model  Accuracy  Precision_0  Recall_0      F1_0  \
0         KNN (k=5, SMOTE)  0.856555     0.882934  0.822124  0.851445   
1  SVM (GridSearch, SMOTE)  0.808551     0.865799  0.730319  0.792309   
2    Random Forest (SMOTE)  0.883685     0.881994  0.885910  0.883948   

   Precision_1  Recall_1      F1_1  
0     0.833573  0.890988  0.861325  
1     0.766792  0.886789  0.822437  
2     0.885392  0.881460  0.883422  


In [ ]:
import os
import joblib
import pickle
from datetime import datetime

# ✅ 動態前置路徑（可根據執行環境設定）
base_dir = "/content/drive/Othercomputers/我的筆記型電腦/ExpertBook/2025/用戶資料集/TrainData/"
model_dir = os.path.join(base_dir, "saved_models")
os.makedirs(model_dir, exist_ok=True)

# 時間戳記版本號
version = datetime.now().strftime("v%Y%m%d_%H%M")

# 儲存為 .joblib
joblib.dump(models["KNN (k=5, SMOTE)"], os.path.join(model_dir, f"knn_model_{version}.joblib"))
joblib.dump(models["SVM (GridSearch, SMOTE)"], os.path.join(model_dir, f"svm_model_{version}.joblib"))
joblib.dump(models["Random Forest (SMOTE)"], os.path.join(model_dir, f"rf_model_{version}.joblib"))

# 儲存為 .pkl
with open(os.path.join(model_dir, f"knn_model_{version}.pkl"), "wb") as f:
    pickle.dump(models["KNN (k=5, SMOTE)"], f)
with open(os.path.join(model_dir, f"svm_model_{version}.pkl"), "wb") as f:
    pickle.dump(models["SVM (GridSearch, SMOTE)"], f)
with open(os.path.join(model_dir, f"rf_model_{version}.pkl"), "wb") as f:
    pickle.dump(models["Random Forest (SMOTE)"], f)

# 儲存轉換器
joblib.dump(scaler, os.path.join(model_dir, f"scaler_{version}.joblib"))
if "le" in locals():
    joblib.dump(le, os.path.join(model_dir, f"label_encoder_{version}.joblib"))

# 建立紀錄檔（log）
with open(os.path.join(model_dir, f"model_log_{version}.txt"), "w") as log:
    log.write("模型版本：" + version + "\n")
    log.write("儲存時間：" + datetime.now().strftime("%Y-%m-%d %H:%M:%S") + "\n")
    # log.write("特徵欄位：" + ", ".join(features) + "\n")
    log.write("模型摘要：\n")
    for row in df_summary.to_dict(orient="records"):
        for k, v in row.items():
            log.write(f"  {k}: {v}\n")
        log.write("\n")

print(f"✅ 模型與轉換器已成功儲存至：{model_dir}（版本：{version}）")


✅ 模型與轉換器已成功儲存至：/content/drive/Othercomputers/我的筆記型電腦/ExpertBook/2025/用戶資料集/TrainData/saved_models（版本：v20250511_1119）
